# Delta Protein Matrix Analysis
## UK Biobank (UKB) Cohort

**Author:** Ximing Ran


## Executive Summary

This notebook calculates **delta protein values** to quantify deviations in protein expression among ALS-affected individuals relative to healthy control baselines in the UK Biobank cohort.

Since UKB provides **baseline cross-sectional data** (one measurement per individual), we use a standard **GAM** (no random effect) fitted on Controls to predict expected protein levels, then compute:

$$\Delta_{protein} = \text{Observed}_{protein} - \text{Expected}_{protein}$$

| Delta Direction | Interpretation |
|-----------------|----------------|
| Positive (+) | Higher than expected; potential risk/disease marker |
| Negative (-) | Lower than expected; potential protective or disease effect |
| Near zero | Expression similar to healthy controls |

**Note:** 1 unit delta = 2-fold change in protein expression (NPX is log2-scale).

**Analysis Pipeline:**
1. Load data
2. Fit GAM baseline model on Controls (age + sex, no random effect)
3. Predict expected values for Affected individuals
4. Calculate delta = observed − expected
5. QC visualizations
6. Export results


## 1. Load Libraries

In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(mgcv)       # GAM
library(progress)
library(knitr)
library(cowplot)

set.seed(2025)
theme_set(
  theme_bw() +
  theme(
    legend.position  = "bottom",
    plot.title       = element_text(face = "bold", size = 14),
    axis.title       = element_text(size = 12),
    axis.text        = element_text(size = 10)
  )
)


## 2. Load Data

### 2.1 Protein Expression Data (Baseline, Wide Format)


In [ ]:
# Baseline protein matrix: rows = participants (eid), columns = proteins
protein_data <- readRDS(here::here("data", "analysis_data", "ukb", "protein_data",
                                    "protein_data_baseline.rds"))

cat("Shape of protein data:", dim(protein_data), "\n")
cat("First 10 columns:", head(colnames(protein_data), 10), "\n")


### 2.2 Visit / Phenotype Data

In [ ]:
visit_info <- read.csv(here::here("data", "analysis_data", "ukb", "visit_info",
                                    "visit_info.csv"), row.names = 1)
# turn the eid to character
visit_info$eid <- as.character(visit_info$eid)
cat("Shape of visit info:", dim(visit_info), "\n")


### 2.3 Differentially Expressed Proteins (from Analysis 1)

In [ ]:
# Load DE results from the linear model analysis
protein_de <- read.csv("../../01-Differentially_Expressed_Proteins/Results/Mix_effect_model_lmer.csv")
# All proteins that passed QC
all_proteins <- setdiff(colnames(protein_data), "eid")

# Significant proteins only
protein_sign <- protein_de %>%
  filter(significant == "Significant") %>%
  pull(Protein)

# Only keep the overlapped proteins
protein_sign <- intersect(protein_sign, all_proteins)

cat("=== Protein Data Summary ===\n")
cat("Total proteins measured:", length(all_proteins), "\n")
cat("Significantly DE proteins:", length(protein_sign), "\n")
cat("Samples in protein matrix:", nrow(protein_data), "\n")


## 3. Define Analysis Groups

- **Control**: Used to fit the GAM baseline model
- **Affected**: Target group for delta calculation


In [ ]:
visit_info_ctrl     <- visit_info %>% filter(Group == "Control")
visit_info_affected <- visit_info %>% filter(Group != "Control")

cat("Control samples:", nrow(visit_info_ctrl), "\n")
cat("Affected samples:", nrow(visit_info_affected), "\n")


In [ ]:
table(visit_info$Group)

## 4. Delta Matrix Calculation

### 4.1 Statistical Model

For each protein, we fit a **GAM** on Control data (no random effect since UKB is cross-sectional):

$$Y_i = \beta_0 + \beta_1 \cdot \text{Sex}_i + f(\text{Age}_i) + \varepsilon_i$$

Where:
- $Y_i$ = Protein expression for subject $i$
- $f(\text{Age})$ = Smooth function of collection age (thin plate regression spline)
- $\varepsilon_i$ = Residual error

We then predict expected values for Affected individuals and compute delta.

### 4.2 Setup Output Directory


In [ ]:
outdir <- here::here("data", "analysis_data", "ukb", "delta_matrix")
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)


### 4.3 Main Loop

In [ ]:
# Use significant proteins for delta calculation
protein_list <- protein_sign

# ── Initialise output containers ──────────────────────────────────────────────
delta_df_long <- data.frame(
  eid          = character(),
  Delta_Value  = numeric(),
  Protein_Name = character(),
  CollAge      = numeric(),
  Sex          = character(),
  GenoGroup    = character(),
  stringsAsFactors = FALSE
)

model_stats <- data.frame(
  Protein         = character(),
  N_Control       = integer(),
  N_Affected      = integer(),
  Model_Converged = logical(),
  R2_Control      = numeric(),
  stringsAsFactors = FALSE
)

for (i in seq_along(protein_list)) {

  protein <- protein_list[i]
  # cat("Processing protein:", protein, "\n")

  suppressWarnings({

    # ── 4a. Build Control data frame ──────────────────────────────────────────
    ctrl_df <- visit_info_ctrl %>%
      left_join(protein_data %>% select(eid, all_of(protein)), by = "eid") %>%
      rename(protein_val = all_of(protein)) %>%
      select(eid, CollAge, Sex, GenoGroup, protein_val) %>%
      filter(!is.na(protein_val)) %>%
      mutate(
        CollAge   = as.numeric(CollAge),
        Sex       = as.factor(Sex),
        GenoGroup = as.factor(GenoGroup)
      )

    # ── 4b. Build Affected data frame ─────────────────────────────────────────
    affected_df <- visit_info_affected %>%
      left_join(protein_data %>% select(eid, all_of(protein)), by = "eid") %>%
      rename(protein_val = all_of(protein)) %>%
      select(eid, CollAge, Sex, GenoGroup, protein_val) %>%
      filter(!is.na(protein_val)) %>%
      mutate(
        CollAge   = as.numeric(CollAge),
        Sex       = factor(Sex,       levels = levels(ctrl_df$Sex)),
        GenoGroup = factor(GenoGroup, levels = levels(ctrl_df$GenoGroup))
      )

    if (nrow(ctrl_df) < 10 || nrow(affected_df) < 3) {
      warning(paste("Skipping", protein, "- insufficient data"))
      return(invisible(NULL))
    }

    mod_ctrl <- tryCatch(
      mgcv::gam(
        protein_val ~ Sex + s(CollAge, k = 5),
        data   = ctrl_df,
        method = "REML"
      ),
      error = function(e) NULL
    )

    if (is.null(mod_ctrl)) {
      model_stats <<- rbind(model_stats, data.frame(
        Protein = protein, N_Control = nrow(ctrl_df),
        N_Affected = nrow(affected_df), Model_Converged = FALSE, R2_Control = NA
      ))
      return(invisible(NULL))
    }

    affected_df$pred_expected <- predict(mod_ctrl, newdata = affected_df, type = "response")
    affected_df$delta         <- affected_df$protein_val - affected_df$pred_expected

    delta_df_long <<- rbind(
      delta_df_long,
      data.frame(
        eid          = as.character(affected_df$eid),
        Delta_Value  = affected_df$delta,
        Protein_Name = protein,
        CollAge      = affected_df$CollAge,
        Sex          = as.character(affected_df$Sex),
        GenoGroup    = as.character(affected_df$GenoGroup),
        stringsAsFactors = FALSE
      )
    )

    model_stats <<- rbind(model_stats, data.frame(
      Protein         = protein,
      N_Control       = nrow(ctrl_df),
      N_Affected      = nrow(affected_df),
      Model_Converged = TRUE,
      R2_Control      = summary(mod_ctrl)$r.sq
    ))

  }) # end suppressWarnings
}

cat("\nDone!\n")


## 5. Results Summary

In [ ]:
cat("=== Delta Calculation Summary ===\n")
cat("Total observations:", nrow(delta_df_long), "\n")
cat("Proteins processed:", length(unique(delta_df_long$Protein_Name)), "\n")
cat("Unique subjects:", length(unique(delta_df_long$eid)), "\n")

cat("\n=== Model Fitting Summary ===\n")
cat("Models converged:", sum(model_stats$Model_Converged, na.rm = TRUE),
    "/", nrow(model_stats), "\n")
cat("Mean R-squared (Control models):",
    round(mean(model_stats$R2_Control, na.rm = TRUE), 3), "\n")

knitr::kable(head(model_stats, 20),
             caption = "Model Fitting Statistics (first 20 proteins)")


## 6. Save Results

In [ ]:
# Long format
write.csv(delta_df_long,
          file.path(outdir, "delta_matrix_long.csv"),
          row.names = FALSE)

# Wide format
delta_df_wide <- delta_df_long %>%
  select(eid, Protein_Name, Delta_Value) %>%
  pivot_wider(names_from = Protein_Name, values_from = Delta_Value)

write.csv(delta_df_wide,
          file.path(outdir, "delta_matrix_wide.csv"),
          row.names = FALSE)

# Model statistics
write.csv(model_stats,
          file.path("Results","1.Delta_matrix", "model_fitting_statistics.csv"),
          row.names = FALSE)

cat("  delta_matrix_long.csv  :", nrow(delta_df_long), "rows\n")
cat("  delta_matrix_wide.csv  :", nrow(delta_df_wide), "rows x",
    ncol(delta_df_wide) - 1, "proteins\n")
cat("  model_fitting_statistics.csv\n")


## 7. Quality Control Visualizations

### 7.1 Delta Distribution Overview


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)

# Overall delta distribution
p1 <- ggplot(delta_df_long, aes(x = Delta_Value)) +
  geom_histogram(aes(y = after_stat(density)),
                 bins = 50, fill = "steelblue", alpha = 0.7) +
  geom_density(color = "darkred", linewidth = 1) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "black") +
  labs(title    = "Distribution of Delta Values",
       subtitle = "Across all proteins and Affected samples",
       x = "Delta (Observed - Expected)",
       y = "Density") +
  theme_minimal()

# Top 20 most variable proteins
top_var_proteins <- delta_df_long %>%
  group_by(Protein_Name) %>%
  summarise(var = var(Delta_Value, na.rm = TRUE), .groups = "drop") %>%
  arrange(desc(var)) %>%
  head(20) %>%
  pull(Protein_Name)

p2 <- delta_df_long %>%
  filter(Protein_Name %in% top_var_proteins) %>%
  ggplot(aes(x = reorder(Protein_Name, Delta_Value, FUN = median),
             y = Delta_Value)) +
  geom_boxplot(fill = "lightblue", outlier.size = 0.5) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "red") +
  coord_flip() +
  labs(title    = "Delta Distribution by Protein",
       subtitle = "Top 20 most variable proteins",
       x = "", y = "Delta Value") +
  theme_minimal()

print(plot_grid(p1, p2, nrow = 1))


### 7.2 Delta Distribution by Sex

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 5)

p_sex <- ggplot(delta_df_long, aes(x = Delta_Value, fill = Sex)) +
  geom_density(alpha = 0.5) +
  geom_vline(xintercept = 0, linetype = "dashed") +
  scale_fill_manual(values = c("Female" = "salmon", "Male" = "steelblue")) +
  labs(title = "Delta Distribution by Sex",
       x = "Delta (Observed - Expected)", y = "Density") +
  theme_minimal()

print(p_sex)


### 7.3 Top 9 Proteins by Mean Absolute Delta

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 10)

top9_proteins <- delta_df_long %>%
  group_by(Protein_Name) %>%
  summarise(abs_mean = abs(mean(Delta_Value, na.rm = TRUE)), .groups = "drop") %>%
  arrange(desc(abs_mean)) %>%
  head(9) %>%
  pull(Protein_Name)

p_top9 <- delta_df_long %>%
  filter(Protein_Name %in% top9_proteins) %>%
  mutate(Protein_Name = factor(Protein_Name, levels = top9_proteins)) %>%
  ggplot(aes(x = CollAge, y = Delta_Value)) +
  geom_point(alpha = 0.5, size = 1.5) +
  geom_smooth(method = "loess", se = TRUE, color = "red") +
  geom_hline(yintercept = 0, linetype = "dashed", color = "gray40") +
  facet_wrap(~Protein_Name, scales = "free_y", ncol = 3) +
  labs(title    = "Delta Values vs Age at Collection (LOESS trend)",
       subtitle = "Top 9 proteins by mean absolute delta",
       x = "Age at Collection",
       y = "Delta Value (Observed - Expected)") +
  theme_minimal() +
  theme(strip.text = element_text(face = "bold"))

print(p_top9)


## 8. All-Protein PDF Export

Generates a multi-page PDF with one delta-vs-age plot per protein (alphabetical order).


In [ ]:
all_proteins_plot <- sort(unique(delta_df_long$Protein_Name))

cat("Total proteins to plot:", length(all_proteins_plot), "\n")

pdf_output_path <- file.path("Results","1.Delta_matrix", "Delta_vs_Age_All_Proteins.pdf")

y_min <- -2
y_max <-  4

pdf(pdf_output_path, width = 10, height = 8)

pb_pdf <- progress_bar$new(
  format = "  Plotting [:bar] :percent | :current/:total proteins",
  total  = length(all_proteins_plot),
  clear  = FALSE
)

for (i in seq_along(all_proteins_plot)) {

  protein      <- all_proteins_plot[i]
  protein_data <- delta_df_long %>% filter(Protein_Name == protein)
  pb_pdf$tick()

  if (nrow(protein_data) == 0) next

  n_samples  <- nrow(protein_data)
  n_subjects <- length(unique(protein_data$eid))
  mean_delta <- round(mean(protein_data$Delta_Value, na.rm = TRUE), 3)
  sd_delta   <- round(sd(protein_data$Delta_Value,   na.rm = TRUE), 3)
  n_below    <- sum(protein_data$Delta_Value < y_min, na.rm = TRUE)
  n_above    <- sum(protein_data$Delta_Value > y_max, na.rm = TRUE)

  # Compute clipped LOESS if enough data
  loess_df <- NULL
  if (n_samples >= 10) {
    loess_fit  <- loess(Delta_Value ~ CollAge, data = protein_data, span = 0.75)
    x_seq      <- seq(min(protein_data$CollAge, na.rm = TRUE),
                      max(protein_data$CollAge, na.rm = TRUE),
                      length.out = 100)
    loess_pred <- predict(loess_fit,
                          newdata = data.frame(CollAge = x_seq),
                          se = TRUE)
    loess_df   <- data.frame(
      CollAge = x_seq,
      fit     = loess_pred$fit,
      se      = loess_pred$se.fit
    ) %>%
      mutate(
        fit_clipped = pmax(pmin(fit, y_max), y_min),
        lower       = pmax(fit - 1.96 * se, y_min),
        upper       = pmin(fit + 1.96 * se, y_max)
      )
  }

  p <- ggplot(protein_data, aes(x = CollAge, y = Delta_Value)) +
    geom_point(alpha = 0.6, size = 2.5) +
    { if (!is.null(loess_df))
        geom_ribbon(data = loess_df,
                    aes(x = CollAge, y = fit_clipped, ymin = lower, ymax = upper),
                    fill = "red", alpha = 0.2, inherit.aes = FALSE) } +
    { if (!is.null(loess_df))
        geom_line(data = loess_df,
                  aes(x = CollAge, y = fit_clipped),
                  color = "red", linewidth = 1.2, inherit.aes = FALSE) } +
    geom_hline(yintercept = 0, linetype = "dashed", color = "gray40", linewidth = 0.8) +
    coord_cartesian(ylim = c(y_min, y_max)) +
    labs(
      title    = "Delta Values vs Age at Collection (LOESS trend)",
      subtitle = paste0("Protein: ", protein,
                        " (", i, " of ", length(all_proteins_plot), ")"),
      x        = "Age at Collection",
      y        = "Delta Value (Observed - Expected)",
      caption  = paste0(
        "N = ", n_samples, " | Mean = ", mean_delta, " | SD = ", sd_delta,
        if (n_below > 0 | n_above > 0)
          paste0(" | Outside range: ", n_below, " below, ", n_above, " above")
        else ""
      )
    ) +
    theme_bw() +
    theme(
      plot.title    = element_text(face = "bold",        size = 16, hjust = 0.5),
      plot.subtitle = element_text(face = "bold.italic", size = 14, hjust = 0.5,
                                   color = "darkblue"),
      plot.caption  = element_text(size = 10, hjust = 0.5, face = "italic"),
      axis.title    = element_text(size = 12),
      axis.text     = element_text(size = 10),
      panel.grid.minor = element_blank()
    )

  print(p)
}

dev.off()

cat("\nPDF saved to:", pdf_output_path, "\n")
cat("Total pages:", length(all_proteins_plot), "\n")


## 9. Protein Summary Statistics

In [ ]:
protein_summary <- delta_df_long %>%
  group_by(Protein_Name) %>%
  summarise(
    N_Samples   = n(),
    N_Subjects  = n_distinct(eid),
    Mean_Delta  = round(mean(Delta_Value, na.rm = TRUE), 3),
    SD_Delta    = round(sd(Delta_Value,   na.rm = TRUE), 3),
    Min_Delta   = round(min(Delta_Value,  na.rm = TRUE), 3),
    Max_Delta   = round(max(Delta_Value,  na.rm = TRUE), 3),
    ABS_Mean    = abs(mean(Delta_Value,   na.rm = TRUE)),
    .groups     = "drop"
  ) %>%
  arrange(desc(ABS_Mean))

write.csv(protein_summary,
          file.path("Results","1.Delta_matrix",  "protein_delta_summary_stats.csv"),
          row.names = FALSE)

cat("Summary saved.\n\n")

protein_summary %>%
  select(-ABS_Mean) %>%
  head(20) %>%
  knitr::kable(caption = "Top 20 Proteins by Absolute Mean Delta")


## 10. Output Files

| File | Description |
|------|-------------|
| `delta_matrix_long.csv` | Tidy format: eid, Delta_Value, Protein_Name, metadata |
| `delta_matrix_wide.csv` | Wide format matrix (samples × proteins) |
| `model_fitting_statistics.csv` | GAM convergence and R² per protein |
| `protein_delta_summary_stats.csv` | Summary statistics per protein |
| `Delta_vs_Age_All_Proteins.pdf` | One plot per protein (alphabetical) |
